In [1]:
# 구글 드라이브 연동
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 59.3 MB/s eta 0:00:00


In [3]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [4]:
# 2. 모델, 인덱스, 문서 로드
MODEL_NAME   = 'jhgan/ko-sbert-nli'
INDEX_PATH   = "/content/drive/MyDrive/RAG/rag_faiss.index"
DOCS_PATH    = "/content/drive/MyDrive/RAG/rag_texts.json"

model = SentenceTransformer(MODEL_NAME)
index = faiss.read_index(INDEX_PATH)

with open(DOCS_PATH, "r", encoding="utf-8") as f:
    docs = json.load(f)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# 3. Retriever 클래스 정의
class Retriever:
    def __init__(self, index, docs, model):
        self.index = index
        self.docs  = docs
        self.model = model

    def retrieve(self, query, top_k=5, filters=None):
        # 3-1. 쿼리 임베딩 & L2 정규화 (IP 인덱스 사용 시 필수)
        q_emb = self.model.encode([query])
        faiss.normalize_L2(q_emb)

        # 3-2. FAISS 검색 (scores 는 코사인 유사도)
        scores, idxs = self.index.search(q_emb, top_k)

        # 3-3. 결과 매핑
        results = []
        for rank, idx in enumerate(idxs[0]):
            results.append({
                "id":       self.docs[idx]["id"],
                "score":    float(scores[0][rank]),
                "context":  self.docs[idx]["context"],
                "metadata": self.docs[idx]["metadata"]
            })

        # 3-4. 메타데이터 필터링 (옵션)
        if filters:
            filtered = []
            for r in results:
                if all(r["metadata"].get(k) == v for k, v in filters.items()):
                    filtered.append(r)
            results = filtered[:top_k]

        return results

In [6]:
# 4. 인스턴스 생성
retriever = Retriever(index=index, docs=docs, model=model)

In [7]:
# 5. 테스트 사용 예시
query   = "이직을 결심한 이유는 무엇인가요?"
filters = {"gender": "FEMALE"}  # 필요한 필터 예시

results = retriever.retrieve(query=query, top_k=5, filters=filters)

for i, r in enumerate(results, 1):
    print(f"[{i}] ID: {r['id']}  |  Score: {r['score']:.4f}")
    print("Context:", r["context"])
    print("Metadata:", r["metadata"], "\n")

[1] ID: entry_6112  |  Score: 0.8272
Context: [질문] 만약 이직을 생각하는 동료를 만나게 되었을 때 뭐라고 말씀하실 건가요 왜 그렇게 말씀하실지 그 이유도 같이 설명해 주세요
[답변] 어 정말 좋은 생각이라고 말해줄 거 같구요. 그거 외에는 딱히 제가 할 수 있는 말은 없다고 생각합니다. 하지만 대신 어 왜 이직을 하고 싶고 이 회사에서 어떤 어려움이 있어서 이직을 하려고 하는지에 대해 이유를 한번 물어볼 것 같긴 해요. 그래서 그 고민이 어려움이 있었던 건지 아니면 좀 더 나은 쪽으로 조건을 보고 가는 건지에 대해 따라서 어 저도 한 번 돌아볼 수 있을 거 같고 그리고 만약에 좀 문제가 있는 시스템들이 있는데 그게 저는 깨닫지 못했고 그 동료는 깨달았기 때문에 나가는 거라면 어 같이 또 일하고 있는 따른 사람들이 나가지 않게 하기 위해서 좀 조치를 취하는 방법을 생각해 볼 것 같습니다.
Metadata: {'occupation': 'BM', 'gender': 'FEMALE', 'experience': 'NEW', 'date': '20230116', 'emotion': [], 'intent_category': ['attitude'], 'summary': '이직의 이유를 물으면 좀 더 나은 조건을 보고 가는 건지에 대해 돌아볼 수 있고, 문제가 있는 시스템들이 있는데 그걸 깨닫지 못하고 그 동료는 깨달았기 때문에 나가는 거라면 같이 일하는 사람들이 나가지 않게 하는 방법을 생각해 볼 것 같습니다.'} 

[2] ID: entry_5960  |  Score: 0.8070
Context: [질문] 만약 이직을 생각하는 동료를 만나게 된다면 지원자님께서는 뭐라고 말씀하시겠습니까 그리고 왜 그렇게 말씀하실지도 설명해 주세요
[답변] 저도 처음으로 지금 이직을 경험하는 거긴 한데요. 제가 이직을 하기 전에 정말 수백 번 고민할 정도로 많은 고민을 했습니다 .그래서 제가 만약에 주변에 이직을 하는 동료가 있다고 하면 저는 여러 가지 조